In [7]:
from dotenv import load_dotenv
load_dotenv()

True

In [15]:
from langchain.text_splitter import Language
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import LanguageParser

repo_path = "/Users/efatsi/Desktop/Projects/franklin/django"

In [16]:
loader = GenericLoader.from_filesystem(
    repo_path,
    glob="**/*",
    suffixes=[".py"],
    exclude=["**/non-utf8-encoding.py"],
    parser=LanguageParser(language=Language.PYTHON, parser_threshold=500),
)
documents = loader.load()

In [17]:
len(documents)

1026

In [18]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, chunk_size=2000, chunk_overlap=200
)
texts = python_splitter.split_documents(documents)
len(texts)

1969

In [20]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

db = Chroma.from_documents(texts, OpenAIEmbeddings(disallowed_special=()))
retriever = db.as_retriever(
    search_type="mmr",  # Also test "similarity"
    search_kwargs={"k": 8},
)

In [21]:
retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x10e73e710>, search_type='mmr', search_kwargs={'k': 8})

In [27]:
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationSummaryMemory
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model_name="gpt-4")
memory = ConversationSummaryMemory(
    llm=llm, memory_key="chat_history", return_messages=True
)
qa = ConversationalRetrievalChain.from_llm(llm, retriever=retriever, memory=memory)

In [23]:
llm.invoke("tell me a knock knock joke")

AIMessage(content='Knock, knock.\n- Who’s there?\nLettuce.\n- Lettuce who?\nLettuce in, it’s too cold out here!')

In [28]:
qa.invoke("explain how taxes are split across payments after making a zeal preview request")

{'question': 'explain how taxes are split across payments after making a zeal preview request',
 'chat_history': [SystemMessage(content='')],
 'answer': "After making a Zeal preview request, taxes are divided across payments proportionally based on each payment's gross amount. This is done through the `_save_preview_data` function.\n\nHere is an outline of how it works:\n\n- The total taxable gross is calculated by summing the gross amounts of all taxable payments.\n- The total net pay is extracted from the preview data.\n- The total taxed net pay is calculated by subtracting the total non-taxable gross from the total net pay.\n- For each taxable payment:\n  - The proportion of the payment's gross to the total taxable gross is calculated.\n  - The payment's preview net, employer taxes, and employee taxes are calculated by multiplying the corresponding values from the preview data by the calculated proportion.\n  - The taxes are also saved in the payment.\n  - Then, the payment is saved

In [29]:
qa.invoke("what code files are necessary to understand the code flow around that functionality")

{'question': 'what code files are necessary to understand the code flow around that functionality',
 'chat_history': [SystemMessage(content="The human asks the AI to explain how taxes are split across payments after making a Zeal preview request. The AI explains that taxes are divided proportionally based on each payment's gross amount through the `_save_preview_data` function. The process includes calculating the total taxable gross and taxed net pay, determining the proportion of each payment's gross to the total, and calculating and saving each payment's net, employer taxes, and employee taxes based on this proportion. This ensures that taxes are distributed across payments in line with their gross amounts.")],
 'answer': 'The following files are necessary to understand the code flow around the functionality of splitting taxes across payments after making a Zeal preview request:\n\n1. `apps.core.models` - This file contains the model definitions for User, Organization, Paycheck, and